# Les 18: AI-agents beveiligen met cryptografische ontvangstbewijzen

## Praktijknotebook

Deze notebook doorloopt vier oefeningen:

1. **Onderteken je eerste ontvangstbewijs** voor een agent-gereedschapsoproep en verifieer het.
2. **Manipuleer het ontvangstbewijs** en zie hoe de verificatie faalt.
3. **Bouw een keten van drie ontvangstbewijzen** en bevestig de ketenintegriteit.
4. **Wikkel een Microsoft Agent Framework-gereedschapsoproep in** zodat elke actie een ontvangstbewijs uitgeeft.

Alle cryptografische primitieve worden geïmporteerd uit goed onderhouden bibliotheken (`pynacl` voor Ed25519, `jcs` voor RFC 8785 canonical JSON, `hashlib` uit de Python-standaardbibliotheek voor SHA-256). De ontvangstbewijslogica zelf is gewone Python die je kunt lezen en aanpassen.

Voer de cellen op volgorde uit. Elke sectie is kort en op zichzelf staand.


## Setup

Installeer de twee afhankelijkheden. Beide hebben permissieve licenties (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Hulpprogramma's

Deze twee hulpprogramma's verwerken base64url-codering (zonder opvulling) en SHA-256-hashing van willekeurige objecten. Ze houden de rest van het notitieboek gericht op de ontvangstlogica zelf.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sectie 1: Onderteken uw eerste ontvangstbewijs

Stel je voor dat onze agent voor **Contoso Travel** net vluchten van Sydney naar Los Angeles voor een klant heeft opgezocht. We willen deze toolaanroep vastleggen als een ondertekend ontvangstbewijs zodat een toekomstige auditor dit kan verifiëren zonder ons te hoeven vertrouwen.

### Stap 1.1: Genereer een ondertekeningssleutel

In productie zou de ondertekeningssleutel van de agent in een hardwarebeveiligingsmodule (HSM), Azure Key Vault of een vergelijkbare beveiligde opslag worden bewaard. Voor deze les genereren we een nieuwe sleutel in het geheugen.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Stap 1.2: Bouw de ontvangstpayload op

De payload bevat alles waarvan we willen dat het ontvangstbewijs bevestigt: wie heeft gehandeld, met welk gereedschap, met welke argumenten, wat er terugkwam, onder welk beleid, en wanneer. We hashen de argumenten en het resultaat in plaats van ze inline op te nemen, zodat het ontvangstbewijs geen gevoelige inhoud lekt.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Stap 1.3: Onderteken en monteer de ontvangstbevestiging

Drie stappen:

1. Canonicaliseer de payload met JCS zodat twee implementaties die dezelfde logische ontvangstbevestiging produceren, byte-identieke bytes opleveren.
2. Hash de canonieke bytes met SHA-256.
3. Onderteken de hash met de Ed25519 private key.

De handtekening wordt vervolgens aan de originele payload toegevoegd om de uiteindelijke ontvangstbevestiging te creëren.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Stap 1.4: Verifieer het ontvangstbewijs

Verificatie keert het proces om. We verwijderen de handtekening, berekenen de canonieke hash opnieuw en controleren de handtekening aan de hand van de openbare sleutel in het ontvangstbewijs.

Een auditor die deze verificatie uitvoert, heeft niets van ons nodig behalve het ontvangstbewijs zelf. Geen dienst om aan te roepen, geen sleutelmap om te raadplegen, geen vertrouwen vereist.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Je zou `Receipt is valid: True` moeten zien. De agent heeft zijn eerste cryptografisch ondertekende auditrecord geproduceerd.


## Sectie 2: Knutselen aan de bon

Het hele punt van bonnetjes is dat ze bestand zijn tegen manipulatie. Laten we dat bewijzen.

We zullen precies één teken van de bon aanpassen en zien dat de verificatie faalt.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Wat is er net gebeurd?

Toen we `policy_id` veranderden, wijzigden de canonieke bytes. De SHA-256-hash van die bytes veranderde. De handtekening (die over de originele hash was) komt niet langer overeen met de nieuwe hash. Verificatie geeft correct `False` terug.

Er is geen manier om een veld van de ontvangstbewijs te wijzigen en het toch te laten verifiëren, tenzij de aanvaller de privésleutel heeft. Zolang de privésleutel zich in een sleutelkluis bevindt en de publieke sleutel is gepubliceerd, is het onmogelijk om manipulatie te verbergen.

Probeer het zelf: wijzig de `tool_name` of `agent_id` of `timestamp` in de bovenstaande cel en voer opnieuw uit. Elke wijziging levert een ongeldig ontvangstbewijs op.


## Sectie 3: Ontvangsten aan elkaar koppelen

Een enkele ontvangst beschermt één actie. De meeste agenten voeren veel acties uit. Om de hele reeks tamper-evident te maken, koppelen we elke ontvangst aan de vorige door de hash van de vorige ontvangst op te nemen in de payload van de nieuwe ontvangst.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Als iemand een ontvangst verwijdert of de volgorde wijzigt, breekt de keten precies op dat punt. Verificatie van een latere ontvangst mislukt omdat de `previous_receipt_hash` niet langer overeenkomt met de werkelijke hash van de voorganger.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Breek nu de keten door te knoeien met het middelste ontvangstbewijs en verifieer opnieuw. Het gemanipuleerde ontvangstbewijs faalt de handtekeningcontrole, EN het volgende ontvangstbewijs faalt de keten-linkcontrole (omdat de `previous_receipt_hash` niet langer overeenkomt met de gewijzigde hash van het middelste ontvangstbewijs).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Ontvangstbewijs 0 verifieert nog steeds (het is niet gewijzigd en heeft geen voorganger om van afhankelijk te zijn). Ontvangstbewijs 1 faalt de handtekeningcontrole omdat we `tool_args_hash` hebben gewijzigd. Ontvangstbewijs 2 faalt de ketenlinkcontrole omdat de `previous_receipt_hash` is berekend op basis van het originele (nu gewijzigde) ontvangstbewijs 1.

Zelfs als een aanvaller het gewijzigde ontvangstbewijs 1 opnieuw ondertekent (wat zij niet kunnen doen zonder de privésleutel), zou de ketenlinkfout in ontvangstbewijs 2 nog steeds de manipulatie blootleggen. Om de wijziging te verbergen, zou de aanvaller elk ontvangstbewijs vanaf het wijzigingspunt opnieuw moeten ondertekenen, wat vereist dat hij de privésleutel bezit.


## Sectie 4: Wikkel een agent tool-aanroep in met ontvangstbevestiging ondertekening

In een echte implementatie wil je niet dat elke agentauteur onthoudt om `make_receipt` aan te roepen. Je wilt dat ontvangstondertekening automatisch gebeurt bij elke tool-aanroep.

Hier is het eenvoudigste patroon: een wrapperklasse die elke aanroepbare toolfunctie neemt en een versie ervan teruggeeft die een ontvangstbewijs genereert. Dit past zich aan elk agentframework aan, inclusief het Microsoft Agent Framework (`agent_framework.azure`).

Als je geen Azure AI Foundry-project hebt opgezet, toont de lokale mock hieronder nog steeds het patroon.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integratie met het Microsoft Agent Framework

De `ReceiptedTool` wrapper hierboven is framework-onafhankelijk. Om deze te gebruiken binnen een agent gebouwd met het Microsoft Agent Framework, registreer je de gewrapte functie als een tool. Een schets (je zou de mock vervangen door een echte Azure AI Foundry tool-registratie):

```python
# Pseudocode die de integratievorm toont.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructies="Je bent een Contoso Travel agent ...",
#     tools=[receipted_lookup],   # de ingepakte tool, niet de ruwe functie
# )
# reactie = agent.run("Vind vluchten van Sydney naar Los Angeles in juni.")
#
# # Na het uitvoeren heeft elke tool-aanroep die de agent deed een ondertekende ontvangstbewijs:
# audit_chain = receipted_lookup.receipts
```

Het agent framework hoeft niets te weten over ontvangstbewijzen. Het ondertekenen van ontvangstbewijzen is rond de tool gewrapt, niet ingebouwd in het framework. Zo voeg je herkomst toe aan bestaande agentcode zonder de agent te herschrijven.


## Samenvatting en uitbreidingsuitdaging

Je hebt:

- Een Ed25519-sleutelpaar gegenereerd.
- Een ontvangstbewijs gebouwd en ondertekend voor een agent-toolaanroep.
- Het ontvangstbewijs offline geverifieerd met alleen de publieke sleutel.
- Een ontvangstbewijs vervalst en gezien dat verificatie faalt.
- Een hash-gekette reeks van drie ontvangstbewijzen gebouwd.
- In het midden van de keten geknoeid en zowel handtekeningfout als ketenlinkfout geconstateerd.
- Een functietool ingepakt met automatische ondertekening van ontvangstbewijzen.

**Uitbreidingsuitdaging.** Breid het schema van het ontvangstbewijs uit met een `request_id` veld (een UUID voor gedistribueerde tracing). Werk `make_receipt` bij om dit op te nemen, en bevestig dat ontvangstbewijzen nog steeds end-to-end verifiëren. Wijzig daarna het veld na het ondertekenen en bevestig dat verificatie faalt. Dit dwingt je te begrijpen hoe elke byte van de canonieke codering bijdraagt aan de handtekening.

**Belangrijke grens.** Ontvangstbewijzen bewijzen drie dingen en alleen die drie: attributie (deze sleutel heeft deze inhoud ondertekend), integriteit (de inhoud is sinds het ondertekenen niet veranderd), en ordening (dit ontvangstbewijs kwam na dat ontvangstbewijs). Ze bewijzen NIET dat de actie van de agent correct was, dat het beleid genoemd in `policy_id` daadwerkelijk is geëvalueerd, of dat de agent elke regel heeft gevolgd. Ontvangstbewijzen vormen een fundament. Governance is het systeem dat je erbovenop bouwt.

Lees de les-README opnieuw met die grens in gedachten. De meest voorkomende fout die teams maken met ontvangstbewijzen is veronderstellen dat "we ontvangstbewijzen hebben" betekent "we worden bestuurd." Dat is niet zo. Ontvangstbewijzen maken het gedrag van een agent controleerbaar. Ze maken het niet correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
